# One-shot, learning-free defect detection on line-scan images

Classical computer-vision pipeline for detecting defects on a bright, horizontal stripe in 200×800 grayscale
line-scan images, designed to work with only a handful of reference images (no deep learning):

1. **Defect-free reconstruction of each image** — histogram equalisation, then every row is replaced by its maximum
   intensity, which erases local dark defects while keeping the stripe profile. Morphological dilation/erosion
   cleans up the profile.
2. **Stripe localisation** — the stripe borders and its centre line are found from the row-mean intensity profile
   (threshold below the peak + local-minimum search); outside the stripe the original pixels are restored.
   A template-correlation variant (`find_horizontal_band_correlation`) is also explored.
3. **Distance features** — MSE, PSNR and SSIM between the image and its defect-free version (plus a patch-wise
   MSE exceedance count).
4. **Decision** — (a) a single threshold on one distance, chosen from the ROC curve, and (b) an RBF-SVM on
   standardised (PSNR/MSE, SSIM) features with an 80/20 train/test split.

**Data:** 14 reference images + 292 labelled test images (good / bad) from a production line — proprietary, not
included. Product images were removed from the outputs; ROC, histogram and feature-space plots are kept.


In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

In [ ]:
import os
from PIL import Image, ImageOps


def load_images_from_drive(folder_path):
  """Loads all images from a specified folder in Google Drive.

  Args:
    folder_path: The path to the folder containing the images in Google Drive.

  Returns:
    A list of PIL Image objects. Returns an empty list if no images are found or if there's an error.
  """
  images = []
  try:
      for filename in os.listdir(folder_path):
          if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
              image_path = os.path.join(folder_path, filename)
              try:
                  img = Image.open(image_path).convert('L')  # Convert to grayscale
                  img = ImageOps.equalize(img)
                  images.append(img)
              except IOError:
                  print(f"Error opening image: {image_path}")
  except FileNotFoundError:
      print(f"Error: Folder not found at {folder_path}")
  except Exception as e:
    print(f"An unexpected error occurred: {e}")

  return images

# Example usage: Replace f'{DATA_ROOT}/your_image_folder' with the actual path
image_folder = f'{DATA_ROOT}/nolearning/product_a/train'
loaded_images = load_images_from_drive(image_folder)

if loaded_images:
    print(f"Loaded {len(loaded_images)} images.")
    # Now you can work with the loaded_images list (e.g., display, process, etc.)
    # For example, to display the first image:
    #loaded_images[0].show()
else:
    print("No images found or an error occurred.")

In [ ]:
import numpy as np

image_arrays = []
images = []
for img in loaded_images:
  image_arrays.append(np.array(img))
  images.append(np.array(img))

# Now image_arrays contains the NumPy arrays of your images.
# You can access them like this:
# image_arrays[0]  # First image
# image_arrays[1]  # Second image and so on...

In [ ]:
import cv2

import matplotlib.pyplot as plt
import numpy as np
# Assuming image_arrays is defined as in the previous code
modified_img = [] # Initialize modified_img as an empty list
for img_array in image_arrays:
  temp_img = img_array.copy()  # Create a copy of the current image
  for i in range(temp_img.shape[0]): # Iterate through each row
    # Row statistic used to build the defect-free profile
    quartile_val = np.mean(temp_img[i])

    # Set every pixel in the row to that value
    temp_img[i] = quartile_val

  # Create kernel
  kernel_size = (7, 7)
  kernel = np.ones(kernel_size, np.uint8)

  # Apply dilation to the temporary image
  temp_img = cv2.dilate(temp_img.astype(np.uint8), kernel, iterations=5)
  temp_img = cv2.erode(temp_img.astype(np.uint8), kernel, iterations=5)

  # Append the modified temporary image to modified_img list
  modified_img.append(temp_img)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
# Assuming 'images' contains the original images and 'image_arrays' contains the modified images

for i in range(len(images)):
    # Create a figure with two subplots (1 row, 2 columns)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # Display the original image in the first subplot
    axes[0].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB), cmap='gray')
    axes[0].set_title(f"Original Image {i+1}")

    # Display the modified image in the second subplot
    axes[1].imshow(modified_img[i].astype(np.uint8), cmap='gray')
    axes[1].set_title(f"Modified Image {i+1}")

    # Show the figure with both images
    plt.show()

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image


def load_images_with_labels(root_dir):
    images = []
    labels = []
    for label, subdir in enumerate(sorted(os.listdir(root_dir))): # Assumes subdirs are 'good' and 'bad'
      subdir_path = os.path.join(root_dir, subdir)
      if os.path.isdir(subdir_path):  # Check if the item in the directory is a subdirectory
        for filename in os.listdir(subdir_path):
          if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            img_path = os.path.join(subdir_path, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) # Load as grayscale
            if img is not None:

                images.append(img)
                labels.append(1-label) # 0 for 'good', 1 for 'bad'
            else:
                print(f"Error loading image: {img_path}")
    return images, labels


# Example usage (replace with your actual path):
root_dir = f'{DATA_ROOT}/nolearning/product_a_v2'
images, labels = load_images_with_labels(root_dir)

print(f"Loaded {len(images)} images.")

# Now you have the images and corresponding labels
# Example: Access the first image and its label
if images:
    print(f"First image shape: {images[0].shape}, label: {labels[0]}")

In [ ]:
import numpy as np
# Assuming 'images' contains the original images from the test set.
defect_free_images = []

for img_array in images:
  temp_img = img_array.copy()  # Create a copy of the current image
  temp_img = cv2.equalizeHist(temp_img)
  for i in range(temp_img.shape[0]): # Iterate through each row
    # Row statistic used to build the defect-free profile


    quartile_val = np.max(temp_img[i])

    # Set every pixel in the row to that value
    temp_img[i] = quartile_val

  # Create kernel
  kernel_size = (7, 7)
  kernel = np.ones(kernel_size, np.uint8)


  # Append the modified temporary image to modified_img list
  defect_free_images.append(temp_img)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def find_horizontal_band_correlation(image1, image2_to_show):
  """
  Computes the 1D cross-correlation signal to find horizontal bands in image1
  after normalizing image1, using np.max on the matchTemplate output,
  and also plots image2_to_show.

  Args:
    image1: The primary input image as a NumPy array (preferably grayscale)
            for correlation computation.
    image2_to_show: A second image to plot alongside image1 for comparison.
                    Should also be a NumPy array (preferably grayscale).

  Returns:
    A NumPy array representing the 1D cross-correlation signal along the
    vertical axis of image1.
  """
  # Ensure image1 is in grayscale
  if len(image1.shape) == 3:
    image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)

  # Ensure image2_to_show is in grayscale for consistent plotting
  if len(image2_to_show.shape) == 3:
      image2_to_show = cv2.cvtColor(image2_to_show, cv2.COLOR_BGR2GRAY)


  image_height, image_width = image1.shape

  # --- START: Image1 Normalization ---
  # Normalize image1 to the full 0-255 range
  normalized_image1 = cv2.normalize(image1, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
  # --- END: Image1 Normalization ---


  # --- Template Creation (White Band) ---
  white_band_height = 7  # Example height, adjust as needed
  template_height = white_band_height

  # Create the white band with the full width of image1 (raw template)
  template = np.ones((white_band_height, image_width), dtype=np.uint8) * 255
  # --- End Template Creation ---


  # Ensure the template height is not greater than image1 height
  if template_height > image_height:
      print("Error: Template height is greater than image height. Cannot perform correlation.")
      return None # Or handle as appropriate

  # Compute the 2D cross-correlation using cv2.TM_CCORR on normalized_image1
  correlation_result = cv2.matchTemplate(normalized_image1.astype(np.float32), template.astype(np.float32), cv2.TM_CCORR)


  # When template width equals image width, the result is a 1D array (N, 1).
  # np.max(..., axis=1) on a (N, 1) array just returns the same values flattened.
  horizontal_correlation_signal = np.max(correlation_result, axis=1)


  # Plotting the original image1, normalized image1, image2_to_show, template, and correlation signal
  plt.figure(figsize=(26, 6)) # Increased figure size to accommodate five plots

  # Subplot for the original image1
  plt.subplot(1, 5, 1)
  plt.imshow(image1, cmap='gray')
  plt.title("Original Image 1")
  plt.xlabel("Column Index")
  plt.ylabel("Row Index")

  # Subplot for the normalized image1
  plt.subplot(1, 5, 2)
  plt.imshow(normalized_image1, cmap='gray')
  plt.title("Normalized Image 1")
  plt.xlabel("Column Index")
  plt.ylabel("Row Index")

  # Subplot for image2_to_show
  plt.subplot(1, 5, 3)
  plt.imshow(image2_to_show, cmap='gray')
  plt.title("Image 2 (Comparison)")
  plt.xlabel("Column Index")
  plt.ylabel("Row Index")


  # Subplot for the template
  plt.subplot(1, 5, 4)
  plt.imshow(template, cmap='gray')
  plt.title("Correlation Template (White Band)")
  plt.xlabel("Column Index")
  plt.ylabel("Row Index")

  # Subplot for the correlation signal
  plt.subplot(1, 5, 5) # Adjusted subplot index
  correlation_y_range = np.arange(len(horizontal_correlation_signal))
  plt.plot(horizontal_correlation_signal, correlation_y_range)
  plt.gca().invert_yaxis()
  plt.xlabel("Correlation Value")
  plt.ylabel("Window Top Row Index")
  plt.title("Correlation Signal (from Image 1)")

  plt.tight_layout()
  plt.show()

  return horizontal_correlation_signal

# Example Usage:
# Assuming 'images[100]' is your primary image for correlation (image1)
# Assuming 'images[101]' is another image you want to display (image2_to_show)
# Make sure you have enough images in your 'images' list
if len(images) > 101: # Check if images[101] exists
    correlation_signal = find_horizontal_band_correlation(defect_free_images[30], images[30])

    if correlation_signal is not None:
        print("Correlation computation complete.")
    else:
        print("Correlation computation failed.")
else:
    print("Not enough images in the 'images' list to run the example.")

Correlation computation complete.


In [ ]:
import matplotlib.pyplot as plt
# Assuming 'images' contains the original images from the test set and defect_free_images is defined as in the previous code.

# Display the first 3 modified images
for i in range(min(3, len(defect_free_images))):  # Limit to the first 3 or fewer
    plt.figure(figsize=(5, 5))  # Adjust figure size as needed
    plt.imshow(defect_free_images[i], cmap='gray')
    plt.title(f"Modified Image {i+1}")
    plt.show()

In [ ]:
import numpy as np

def find_white_stripe_borders(image):
    """Finds the borders of a white stripe in an image.

    Args:
        image: The input image as a NumPy array.
        tolerance: The intensity difference allowed for a pixel to be
                   considered part of the stripe.

    Returns:
        A tuple containing the upper and lower border indices of the stripe.
    """
    row_means = np.mean(image, axis=1)
    center_row = np.argmax(row_means)
    threshold = row_means[center_row]-120 # Intensity of the center row (stripe)

    # Find upper border
    upper_border = 0
    while upper_border < center_row  and row_means[upper_border] <  threshold :
        upper_border += 1

    # Find lower border
    lower_border = image.shape[0] - 1
    while lower_border > center_row and row_means[lower_border]  < threshold :
        lower_border -= 1
    # 2. Find center row using local minimum with intensity change (upward and downward search)
    search_start = upper_border
    search_end = lower_border

    # Iteratively refine the search area
    while True:
        if search_start == search_end:
                center_row=upper_border
                break   # Or handle this case as you prefer

        local_min_center = search_start + np.argmin(row_means[search_start:search_end])

        # Check if local minimum is at a border
        if local_min_center <= search_start +5 or local_min_center >= search_end-5 :
            # Adjust search range to exclude the border point
            if local_min_center <= search_start + 5:
                search_start += 1
            else:
                search_end -= 1

            # If search range becomes empty, break the loop (no suitable local minimum found)
        else:
            # Local minimum is not at a border, so we're done
            center_row=local_min_center
            break


    # Return the borders
    return (upper_border, lower_border,center_row)

In [ ]:
import matplotlib.pyplot as plt
# Assuming 'images' contains the original images and 'defect_free_images' contains the modified images
# and find_white_stripe_borders function is defined as in the previous code.
upper_borders = []
lower_borders = []
center_rows = []
for i in range(len(images)):
    original_image = images[i].copy()

    # Get a copy of the modified image to avoid changing the original list
    modified_image = defect_free_images[i].copy()

    upper_border, lower_border,center_row = find_white_stripe_borders(modified_image)
    upper_borders.append(upper_border)
    lower_borders.append(lower_border)
    center_rows.append(center_row)
    # Check if borders were found
    if upper_border != -1 and lower_border != -1:
        # Replace areas outside borders with original image
        modified_image[:upper_border, :] = original_image[:upper_border, :]
        modified_image[lower_border:, :] = original_image[lower_border:, :]
        # Get the shape of the border region slice
        '''border_shape = modified_image[74:125, :].shape

        white_border = np.ones(border_shape) * np.max(np.mean(original_image[upper_border:lower_border, :], axis=1))

        modified_image[74:125, :] = white_border


        '''

        border_shape = modified_image[upper_border:lower_border, :].shape

        white_border = np.ones(border_shape) * np.max(np.mean(original_image[upper_border:lower_border, :], axis=1))

        modified_image[upper_border:lower_border, :] = white_border





        modified_image[center_row-3:center_row+3,:] =original_image[center_row-3:center_row+3,:]
    # Update the modified image in the list
    defect_free_images[i] = modified_image

In [ ]:
import numpy as np
import math
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
def compute_image_distance(image1, image2, metric='mse',lower=199,upper=0):
    """Computes the distance between two images based on the chosen metric.

    Args:
        image1: The first image as a NumPy array.
        image2: The second image as a NumPy array.
        metric: The distance metric to use ('psnr', 'ssim', 'mse'). Defaults to 'mse'.

    Returns:
        The computed distance between the two images. Returns -1 if an error occurs
        or if the metric is not supported.
    """
    if metric.lower() == 'patch':
        # Get image dimensions
        height, width = image1.shape
        grid_size = (50, 200)
        # Calculate grid cell dimensions
        cell_height = height // grid_size[0]
        cell_width = width // grid_size[1]

        # --- Optimization: Calculate squared difference outside the loop ---
        total_diff = (image1.astype(np.int32) - image2.astype(np.int32))**2  # Calculate squared differences

        # Initialize overpass counter
        overpass_count = 0
        threshold = 1500  # Adjust threshold as needed for MSE

        # Iterate through grid cells
        for i in range(grid_size[0]):
            for j in range(grid_size[1]):
                # Extract grid cell from the total difference
                cell_diff = total_diff[i * cell_height:(i + 1) * cell_height,
                                        j * cell_width:(j + 1) * cell_width]

                # Calculate MSE for the patch
                diff = np.mean(cell_diff)

                if diff > threshold:
                    overpass_count += 1

        return overpass_count

    if metric.lower() == 'mse':
        diff = (image1.astype(np.int32) - image2.astype(np.int32))**2
        mse = np.max(diff)
        return mse
    elif metric.lower() == 'psnr':
        psnr_score = psnr(image1, image2, data_range=255)
        return psnr_score
    elif metric.lower() == 'ssim':
        # Use OpenCV's built-in SSIM function
        ssim_score = ssim(image1, image2)
        return ssim_score
    else:
        print(f"Error: Unsupported metric '{metric}'.")
        return -1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def analyze_image_distances(original_images, modified_images, labels,lower_borders,upper_borders):
    distances = []
    for i in range(len(original_images)):
        distance = compute_image_distance(original_images[i], modified_images[i],'mse',lower_borders[i],upper_borders[i])
        if np.isinf(distance):
            distance = 100
        distances.append(distance)

    # Separate distances based on labels
    distances_1 = [distances[i] for i in range(len(distances)) if labels[i] == 1]
    distances_0 = [distances[i] for i in range(len(distances)) if labels[i] == 0]
    # Replace infinite values with a large finite value
    distances_1 = [d if np.isfinite(d) else 100 for d in distances_1]  # Replace inf with 100
    distances_0 = [d if np.isfinite(d) else 100 for d in distances_0]  # Replace inf with 100

    # Histogram of distances with colored labels
    plt.figure(figsize=(8, 6))
    plt.hist(distances_0, bins=20, color='blue', alpha=0.7, label='Label 1')
    plt.hist(distances_1, bins=20, color='red', alpha=0.7, label='Label 0')
    plt.xlabel("Image Distance")
    plt.ylabel("Frequency")
    plt.title("Distribution of Image Distances by Label")
    plt.legend()  # Add a legend to show color mapping
    plt.show()
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(labels, distances)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()

    # Find best threshold based on Youden's J statistic
    j_statistic = tpr - fpr
    best_threshold_index = np.argmax(j_statistic)
    best_threshold = thresholds[best_threshold_index]
    print(f"Best Threshold: {best_threshold}")

    # Compute accuracy
    predictions = np.array(distances) > best_threshold
    accuracy = np.mean(predictions == labels)
    print(f"Accuracy at best threshold: {accuracy}")
    return best_threshold,predictions
# Example Usage (assuming 'images', 'defect_free_images', and 'labels' are defined):
best_threshold ,predictions  =analyze_image_distances(images, defect_free_images, labels,lower_borders,upper_borders)

Best Threshold: 8281.0
Accuracy at best threshold: 0.9794520547945206


In [ ]:
def show_sample_images(images, modified_image, labels, predictions,lower_borders,upper_borders, num_samples_per_class=25):
    """Shows misclassified sample images, distances, and predicted labels for each class.

    Args:
        images: Original images.
        defect_free_images: Modified images.
        labels: True labels.
        predictions: Predicted labels.
        num_samples_per_class: Number of samples to display per class.
    """

    misclassified_indices_0 = [i for i, (pred, true) in enumerate(zip(predictions, labels)) if pred != true and true == 0]
    misclassified_indices_1 = [i for i, (pred, true) in enumerate(zip(predictions, labels)) if pred != true and true == 1]


    # Limit to num_samples_per_class if more misclassified samples are found
    samples_0 = misclassified_indices_0[:num_samples_per_class]
    samples_1 = misclassified_indices_1[:num_samples_per_class]
    print(f"Misclassified Samples for Class 0 (good): {len(samples_0)} found, showing {min(num_samples_per_class, len(samples_0))}")
    for i in samples_0:
        distance = compute_image_distance(images[i], modified_image[i],'mse',lower_borders[i],upper_borders[i])
        predicted_label = predictions[i]  # Get predicted label from predictions

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(images[i], cmap='gray')
        axes[0].set_title(f"Original Image (Label: {labels[i]})")

        axes[1].imshow(defect_free_images[i], cmap='gray')
        axes[1].set_title(f"Modified Image\nDistance: {distance:.4f}, Predicted: {predicted_label}")
        plt.show()
    print(f"Misclassified Samples for Class 1 (bad): {len(samples_1)} found, showing {min(num_samples_per_class, len(samples_1))}")
    for i in samples_1:
        distance = compute_image_distance(images[i], modified_image[i],'mse',lower_borders[i],upper_borders[i])
        predicted_label = predictions[i]  # Get predicted label from predictions

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(images[i], cmap='gray')
        axes[0].set_title(f"Original Image (Label: {labels[i]})")

        axes[1].imshow(defect_free_images[i], cmap='gray')
        axes[1].set_title(f"Modified Image\nDistance: {distance:.4f}, Predicted: {predicted_label}")
        plt.show()
print(best_threshold)
show_sample_images(images, defect_free_images, labels,predictions ,lower_borders,upper_borders)

8281.0
Misclassified Samples for Class 0 (good): 4 found, showing 4


Misclassified Samples for Class 1 (bad): 2 found, showing 2


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Import for 3D plotting
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr


def analyze_image_distances(original_images, modified_images, labels):

    distances_mse = []
    distances_ssim = []
    distances_psnr = []  # List to store PSNR values
    for i in range(len(original_images)):
        distance_mse = compute_image_distance(original_images[i], modified_images[i], metric='mse')
        distance_ssim = compute_image_distance(original_images[i], modified_images[i], metric='ssim')
        distance_psnr = compute_image_distance(original_images[i], modified_images[i], metric='psnr')  # Calculate PSNR
        distances_mse.append(distance_mse)
        distances_ssim.append(distance_ssim)
        distances_psnr.append(distance_psnr)  # Append PSNR value


    # Reverse labels for MSE (if needed for thresholding)
    reversed_labels_mse = [1 - label for label in labels]

    # Create a figure with subplots
    fig = plt.figure(figsize=(15, 10))

    # 3D Scatter Plot
    ax1 = fig.add_subplot(2, 2, 1, projection='3d')
    scatter1 = ax1.scatter(distances_mse, distances_ssim, distances_psnr,
                           c=reversed_labels_mse, cmap='viridis')
    ax1.set_xlabel("MSE Distance")
    ax1.set_ylabel("SSIM Similarity")
    ax1.set_zlabel("PSNR")
    ax1.set_title("3D Scatter Plot")
    fig.colorbar(scatter1, ax=ax1, label="Reversed Label for MSE (0: good, 1: bad)")

    # 2D Scatter Plots
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.scatter(distances_mse, distances_ssim, c=reversed_labels_mse, cmap='viridis')
    ax2.set_xlabel("MSE Distance")
    ax2.set_ylabel("SSIM Similarity")
    ax2.set_title("MSE vs. SSIM")

    ax3 = fig.add_subplot(2, 2, 3)
    ax3.scatter(distances_mse, distances_psnr, c=reversed_labels_mse, cmap='viridis')
    ax3.set_xlabel("MSE Distance")
    ax3.set_ylabel("PSNR")
    ax3.set_title("MSE vs. PSNR")

    ax4 = fig.add_subplot(2, 2, 4)
    ax4.scatter(distances_ssim, distances_psnr, c=reversed_labels_mse, cmap='viridis')
    ax4.set_xlabel("SSIM Similarity")
    ax4.set_ylabel("PSNR")
    ax4.set_title("SSIM vs. PSNR")

    plt.tight_layout()  # Adjust layout to prevent overlapping
    plt.show()


analyze_image_distances(images, defect_free_images, labels)

In [ ]:
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler # Import StandardScaler

def analyze_image_distances_svm(original_images, modified_images, labels):
    distances_psnr = []
    distances_ssim = []
    for i in range(len(original_images)):
        distance_psnr = compute_image_distance(original_images[i], modified_images[i], metric='mse')
        distance_ssim = compute_image_distance(original_images[i], modified_images[i], metric='ssim')

        # Replace infinite values with a large finite value (e.g., 100)
        if np.isinf(distance_psnr):
            distance_psnr = 100
        if np.isinf(distance_ssim):  # Check for infinity in SSIM as well
            distance_ssim = 1  # SSIM range is typically [-1, 1]

        distances_psnr.append(distance_psnr)
        distances_ssim.append(distance_ssim)

    # Create the dataset
    X = np.column_stack((distances_psnr, distances_ssim))
    y = np.array(labels)

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Feature Scaling (Standardization) - Added
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


    # Train an SVM model with RBF kernel - Changed kernel
    clf = SVC(kernel='rbf', random_state=0).fit(X_train, y_train)  # Using RBF kernel

    # Make predictions on the test set
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Accuracy of the SVM classifier: {accuracy}")

    # Plot the data and decision boundary - Added detailed plotting
    plt.figure(figsize=(10, 6))
    plt.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1],
                alpha=0.5, label='Class 0 (bad)', marker='o')
    plt.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1],
                alpha=0.5, label='Class 1 (good)', marker='x')
    plt.xlabel('mse (Standardized)')
    plt.ylabel('SSIM (Standardized)')
    plt.title('SVM with RBF Kernel: Scatter Plot with Decision Boundary') # Updated title
    plt.legend()

    ## Plot decision boundary using a filled contour
    ax = plt.gca() # Get current axes
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Create grid to evaluate model
    xx = np.linspace(xlim[0], xlim[1], 100) # Increased points for smoother contour
    yy = np.linspace(ylim[0], ylim[1], 100) # Increased points for smoother contour
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T
    Z = clf.decision_function(xy).reshape(XX.shape)

    # Plot decision boundary and margins
    ax.contourf(XX, YY, Z, levels=[-1, 0, 1], alpha=0.5,
               colors=['blue', 'k', 'red'])  # Customize colors as needed

    # Plot support vectors - Included from original code
    plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100,
                linewidth=1, facecolors='none', edgecolors='k')


    plt.show()

# Example Usage (assuming 'images', 'defect_free_images', and 'labels' are defined):
# You would call this function after loading and processing your images:
# analyze_image_distances_svm(images, defect_free_images, labels)

analyze_image_distances_svm(images, defect_free_images, labels)

Accuracy of the SVM classifier: 1.0


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


def analyze_image_distances_svm_with_return(original_images, modified_images, labels):
    """
    Calculates image distances using PSNR and SSIM, splits data, trains an SVM classifier,
    and returns the necessary components for visualizing misclassified samples.
    Does NOT plot the decision boundary.

    Args:
        original_images: List of original images.
        modified_images: List of modified images.
        labels: True labels.

    Returns:
        A tuple containing:
        original_images_test: List of original images from the test set.
        modified_images_test: List of modified images from the test set.
        y_test: True labels for the test set.
        y_pred: Predicted labels for the test set from the SVM classifier.
        scaler: The fitted StandardScaler used for feature scaling.
        clf: The trained SVM classifier object.
    """
    distances_psnr = []
    distances_ssim = []
    for i in range(len(original_images)):
        distance_psnr = compute_image_distance(original_images[i], modified_images[i], metric='psnr')
        distance_ssim = compute_image_distance(original_images[i], modified_images[i], metric='ssim')

        # Handle potential infinite or NaN values
        if np.isinf(distance_psnr) or np.isneginf(distance_psnr):
            distance_psnr = 100 # Replace with a large finite value
        if np.isnan(distance_ssim):
            distance_ssim = 0 # Replace NaN with 0 or another appropriate value

        distances_psnr.append(distance_psnr)
        distances_ssim.append(distance_ssim)

    X = np.column_stack((distances_psnr, distances_ssim))
    y = np.array(labels)

    # Split data and corresponding images into training and testing sets
    X_train, X_test, y_train, y_test, original_images_train, original_images_test, modified_images_train, modified_images_test = train_test_split(
        X, y, original_images, modified_images, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    clf = SVC(kernel='rbf', random_state=0).fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy of the SVM classifier: {accuracy}")

    # Return necessary outputs, including the test sets of original and modified images
    return original_images_test, modified_images_test, y_test, y_pred, scaler, clf

def show_svm_misclassified_samples(original_images_test, modified_images_test, y_test, y_pred, scaler, clf, num_samples_to_display_per_class=5):
    """
    Shows misclassified sample images from the test set based on SVM predictions.

    Args:
        original_images_test: List of original images from the test set.
        modified_images_test: List of modified images from the test set.
        y_test: True labels for the test set.
        y_pred: Predicted labels for the test set from the SVM classifier.
        scaler: The fitted StandardScaler used for feature scaling.
        clf: The trained SVM classifier object.
        num_samples_to_display_per_class: Maximum number of misclassified samples to display from each class.
    """

    # Find misclassified indices for each class in the test set
    misclassified_indices_class0 = [i for i, (pred, true) in enumerate(zip(y_pred, y_test)) if pred != true and true == 0]
    misclassified_indices_class1 = [i for i, (pred, true) in enumerate(zip(y_pred, y_test)) if pred != true and true == 1]

    print(f"Total misclassified samples for Class 0 (True Label 0): {len(misclassified_indices_class0)}. Showing up to {num_samples_to_display_per_class} samples.")
    # Display misclassified samples for Class 0 (True Label 0)
    for i in misclassified_indices_class0[:num_samples_to_display_per_class]:
        original_img = original_images_test[i]
        modified_img = modified_images_test[i]
        true_label = y_test[i]
        predicted_label = y_pred[i]

        # Calculate original (unscaled) mse and SSIM for display
        original_psnr = compute_image_distance(original_img, modified_img, metric='mse')
        original_ssim = compute_image_distance(original_img, modified_img, metric='ssim')

        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(original_img, cmap='gray')
        plt.title(f"Original Image\nTrue Label: {true_label}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(modified_img, cmap='gray')
        plt.title(f"Modified Image\nPredicted Label: {predicted_label}\nPSNR: {original_psnr:.2f}, SSIM: {original_ssim:.2f}")
        plt.axis('off')
        plt.show()

    print(f"\nTotal misclassified samples for Class 1 (True Label 1): {len(misclassified_indices_class1)}. Showing up to {num_samples_to_display_per_class} samples.")
    # Display misclassified samples for Class 1 (True Label 1)
    for i in misclassified_indices_class1[:num_samples_to_display_per_class]:
        original_img = original_images_test[i]
        modified_img = modified_images_test[i]
        true_label = y_test[i]
        predicted_label = y_pred[i]

        original_psnr = compute_image_distance(original_img, modified_img, metric='mse')
        original_ssim = compute_image_distance(original_img, modified_img, metric='ssim')

        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(original_img, cmap='gray')
        plt.title(f"Original Image\nTrue Label: {true_label}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(modified_img, cmap='gray')
        plt.title(f"Modified Image\nPredicted Label: {predicted_label}\nPSNR: {original_psnr:.2f}, SSIM: {original_ssim:.2f}")
        plt.axis('off')
        plt.show()

# --- How to use the new functions ---

# 1. Run the modified analyze_image_distances_svm function to get the results
original_images_test, modified_images_test, y_test, y_pred, scaler, clf = analyze_image_distances_svm_with_return(images, defect_free_images, labels)

# 2. Call the new function to display misclassified samples
show_svm_misclassified_samples(original_images_test, modified_images_test, y_test, y_pred, scaler, clf, num_samples_to_display_per_class=5)

Accuracy of the SVM classifier: 0.9152542372881356
Total misclassified samples for Class 0 (True Label 0): 2. Showing up to 5 samples.



Total misclassified samples for Class 1 (True Label 1): 3. Showing up to 5 samples.
